The created splits are logged in the files in the same directory (train/val/test_split.txt)

In [ ]:
import os
import glob
import random
import shutil
from pathlib import Path

ROOT_DIR = "/workspace/mnt-data/SEN12"
OUTPUT_DIR = "/workspace/mnt-data/SEN12-splits"
TARGET_COUNTS = {"train": 11610, "val": 3582, "test": 5980}
SEED = 42
random.seed(SEED)

In [ ]:
def find_seasons(root_dir):
    return [
        Path(root_dir) / name
        for name in os.listdir(root_dir)
        if (Path(root_dir) / name).is_dir() and "ROIs" in name
    ]


def find_matching_patches(s1_dir, s2_dir):
    patch_pairs = []

    for s1_patch in glob.glob(str(s1_dir / "*.png")):
        s1_patch = Path(s1_patch)

        s2_filename = s1_patch.name.replace("s1_", "s2_")
        s2_patch = s2_dir / s2_filename

        if s2_patch.exists():
            patch_pairs.append((s1_patch, s2_patch))

    return patch_pairs


def collect_scenes(root_dir):
    scenes = []

    for season_dir in find_seasons(root_dir):
        s1_scene_dirs = sorted(season_dir.glob("s1_*"))

        for s1_dir in s1_scene_dirs:
            scene_id = s1_dir.name.split("_")[-1]
            s2_dir = season_dir / f"s2_{scene_id}"

            if not s2_dir.is_dir():
                continue

            patch_pairs = find_matching_patches(s1_dir, s2_dir)

            if patch_pairs:
                scenes.append(patch_pairs)

    return scenes


def split_scenes_into_sets(scenes, target_counts):
    random.shuffle(scenes)

    splits = {"test": [], "val": [], "train": []}

    scene_index = 0

    for split_name in ["test", "val", "train"]:
        target_count = target_counts[split_name]
        selected_patches = []

        while len(selected_patches) < target_count and scene_index < len(scenes):
            scene_patches = scenes[scene_index]
            scene_index += 1

            patches_still_needed = target_count - len(selected_patches)

            if len(scene_patches) <= patches_still_needed:
                selected_patches.extend(scene_patches)
            else:
                random.shuffle(scene_patches)
                selected_patches.extend(scene_patches[:patches_still_needed])

        splits[split_name] = selected_patches

    return splits


def write_split_list(output_dir, split_name, patch_pairs):
    output_file = output_dir / f"{split_name}_split.txt"

    with output_file.open("w") as file:
        for s1_path, s2_path in patch_pairs:
            file.write(f"{s1_path},{s2_path}\n")


def copy_patch_pairs(output_dir, split_name, patch_pairs):
    split_dir = output_dir / split_name
    input_dir = split_dir / "input"
    target_dir = split_dir / "target"

    input_dir.mkdir(parents=True, exist_ok=True)
    target_dir.mkdir(parents=True, exist_ok=True)

    for s1_path, s2_path in patch_pairs:
        s1_destination = input_dir / s1_path.name
        s2_destination = target_dir / s2_path.name

        if not s1_destination.exists():
            shutil.copy2(s1_path, s1_destination)

        if not s2_destination.exists():
            shutil.copy2(s2_path, s2_destination)


def save_splits(output_dir, splits):
    output_dir.mkdir(parents=True, exist_ok=True)

    for split_name, patch_pairs in splits.items():
        write_split_list(output_dir, split_name, patch_pairs)
        copy_patch_pairs(output_dir, split_name, patch_pairs)


def create_sen12_split(root_dir, output_dir, target_counts):
    scenes = collect_scenes(root_dir)

    if not scenes:
        raise RuntimeError("Keine passenden SEN12-Szenen gefunden.")

    splits = split_scenes_into_sets(scenes, target_counts)
    save_splits(output_dir, splits)

    for split_name, patch_pairs in splits.items():
        print(f"{split_name}: {len(patch_pairs)} Patch-Paare")


if __name__ == "__main__":
    create_sen12_split(
        root_dir=Path(ROOT_DIR),
        output_dir=Path(OUTPUT_DIR),
        target_counts=TARGET_COUNTS,
    )

In [ ]:
val_folder = Path(f"{OUTPUT_DIR}/val/target")
train_folder = Path(f"{OUTPUT_DIR}/train/target")
test_folder = Path(f"{OUTPUT_DIR}/test/target")

for path in val_folder.rglob("*_s2_*"):
    if path.is_file():
        matching_name = path.name.replace("_s2_", "_s1_")
        path.rename(path.with_name(matching_name))

for path in train_folder.rglob("*_s2_*"):
    if path.is_file():
        matching_name = path.name.replace("_s2_", "_s1_")
        path.rename(path.with_name(matching_name))

for path in test_folder.rglob("*_s2_*"):
    if path.is_file():
        matching_name = path.name.replace("_s2_", "_s1_")
        path.rename(path.with_name(matching_name))